# 第8章 数えずに学ぶ：ニューラルbigram

確率の表を勾配降下法で「学習」させたら、数えた答えに近づいていった

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/08-neural-bigram.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備


In [ ]:
import re, math, urllib.request
import torch
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]
print(V, train_data.shape, val_data.shape)

In [ ]:
xs, ys = train_data[:-1], train_data[1:]     # 入力（今の文字）と正解（次の文字）
vxs, vys = val_data[:-1], val_data[1:]
print("入力：", xs[:8].tolist())
print("正解：", ys[:8].tolist())
print("文字で見ると：", [itos[i] for i in xs[:8].tolist()], "→", [itos[i] for i in ys[:8].tolist()])

## モデル：中身がでたらめな表


In [ ]:
W = torch.zeros((V, V), requires_grad=True)
print(W.shape, "パラメータの数：", W.numel())

In [ ]:
x = torch.tensor([stoi["先"]])
logits = W[x]                       # 「先」の行＝次の文字の点数（ロジット）
probs = F.softmax(logits, dim=1)
print(logits.shape, probs.shape)
print("「生」の確率：", probs[0, stoi["生"]].item(), " ＝ 1/V：", 1 / V)

## 損失：交差エントロピー


In [ ]:
def loss_by_hand(W, x, y):
    logits = W[x]
    probs = F.softmax(logits, dim=1)
    p_correct = probs[torch.arange(len(y)), y]   # 各行の「正解の文字」の確率
    return -p_correct.log().mean()

print("手作りの損失：", loss_by_hand(W, xs[:1000], ys[:1000]).item())
print("F.cross_entropy：", F.cross_entropy(W[xs[:1000]], ys[:1000]).item())
print("log V：", math.log(V))

## 学習：ミニバッチで坂を下る


In [ ]:
g = torch.Generator().manual_seed(0)
W = torch.zeros((V, V), requires_grad=True)
lr = 50
history = []

for step in range(3001):
    ix = torch.randint(0, len(xs), (1024,), generator=g)   # ランダムに1024問選ぶ（(1024,) は形の指定。要素1つでもカンマが必要）
    loss = F.cross_entropy(W[xs[ix]], ys[ix])               # ① 損失
    W.grad = None                                            # （勾配をリセット）
    loss.backward()                                          # ② 勾配
    with torch.no_grad():
        W -= lr * W.grad                                     # ③ 更新
    if step % 250 == 0:
        with torch.no_grad():
            val_loss = F.cross_entropy(W[vxs], vys).item()
        history.append((step, loss.item(), val_loss))
        if step % 500 == 0:
            print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss:.3f}")

## 学習曲線を描く


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

steps = [h[0] for h in history]
plt.figure(figsize=(6, 3.5))
plt.plot(steps, [h[1] for h in history], label="訓練（ミニバッチ）")
plt.plot(steps, [h[2] for h in history], label="検証")
plt.axhline(3.581, color="gray", linestyle="--", label="数えたbigram（第5章）")
plt.xlabel("ステップ")
plt.ylabel("損失")
plt.ylim(3, 8)
plt.legend()
plt.title("ニューラルbigramの学習曲線")
plt.show()

## 学習した表の中身を見る


In [ ]:
with torch.no_grad():
    p = F.softmax(W[stoi["先"]], dim=0)
top = torch.topk(p, 5)
print("学習したモデル：「先」の次は…")
for prob_, i in zip(top.values, top.indices):
    print(f"  {itos[i.item()]!r} {prob_.item():.3f}")

## 生成してみる


In [ ]:
g = torch.Generator().manual_seed(1)
ix = stoi["私"]
out = ["私"]
with torch.no_grad():
    for _ in range(60):
        p = F.softmax(W[ix], dim=0)
        ix = torch.multinomial(p, 1, generator=g).item()
        out.append(itos[ix])
print("".join(out))